# 文本预处理
:label:`sec_text_preprocessing`

对于序列数据处理问题，我们在 :numref:`sec_sequence`中
评估了所需的统计工具和预测时面临的挑战。
这样的数据存在许多种形式，文本是最常见例子之一。
例如，一篇文章可以被简单地看作一串单词序列，甚至是一串字符序列。
本节中，我们将解析文本的常见预处理步骤。
这些步骤通常包括：

1. 将文本作为字符串加载到内存中。
1. 将字符串拆分为词元（如单词和字符）。
1. 建立一个词表，将拆分的词元映射到数字索引。
1. 将文本转换为数字索引序列，方便模型操作。


In [1]:
import collections
import re
from d2l import torch as d2l

## 导入的包及其作用

上面三行导入了本节所需的库，可以分成「Python 标准库」和「教程工具包」两类：

| 语句 | 来源 | 作用 |
| --- | --- | --- |
| `import collections` | Python 标准库 | 这里只用其中的 `Counter`，在 `count_corpus` 中统计每个词元出现的频率 |
| `import re` | Python 标准库 | 正则表达式模块，用 `re.sub` 清洗文本（把非字母字符替换成空格） |
| `from d2l import torch as d2l` | 第三方教程库 | 导入《动手学深度学习》配套工具包的 PyTorch 版本，并提供数据下载等辅助函数 |

### 1. `import collections`

`collections` 是 Python 标准库中的容器模块，还提供 `defaultdict`、`OrderedDict`、`deque` 等常用容器。本节只用到其中一个类：

- `collections.Counter`：在 `count_corpus` 中统计词元频率，它相当于「字典 + 计数」的结合体。

```python
return collections.Counter(tokens)
```

喂给它一个可迭代对象（这里是词元列表），它会自动返回每个元素出现的次数：

```python
Counter(['a', 'b', 'a'])   # Counter({'a': 2, 'b': 1})
```

后续 `Vocab` 就是靠它按频率给词元排序，并为每个词元分配数字索引。

### 2. `import re`

`re` 是 Python 标准库的正则表达式模块。本节只用到其中一个函数：

```python
re.sub('[^A-Za-z]+', ' ', line)
```

- `re.sub(模式, 替换文本, 源字符串)`：找出源字符串中所有匹配「模式」的部分，统一替换后返回**新字符串**。
- 这里把「一个或多个非字母字符（数字、标点、空白等）」替换成单个空格，从而完成文本清洗。

### 3. `from d2l import torch as d2l`

`d2l` 是《动手学深度学习》作者编写的教程配套工具包，把全书中反复用到的功能封装好了，避免每章重复造轮子。

- `torch` 表示「基于 PyTorch 后端」的版本（同类还有 `from d2l import tensorflow as d2l`）。
- `as d2l` 是起别名，之后统一用 `d2l.xxx` 调用。
- 本节用到它的：
  - `d2l.DATA_URL`：数据集的下载地址前缀；
  - `d2l.download('time_machine')`：先查本地缓存，没有就下载，返回本地文件路径；
  - `d2l.DATA_HUB`：登记数据集的全局字典（键名 → 下载地址 + SHA-1 校验和）。

### 补充：`#@save` 是什么

后面代码中出现 `#@save`，这是 d2l 的专用标记，表示「请把下面这个函数/类保存进 d2l 包」，方便全书统一调用，**不影响代码运行**。

## 读取数据集

首先，我们从H.G.Well的[时光机器](https://www.gutenberg.org/ebooks/35)中加载文本。
这是一个相当小的语料库，只有30000多个单词，但足够我们小试牛刀，
而现实中的文档集合可能会包含数十亿个单词。
下面的函数(**将数据集读取到由多条文本行组成的列表中**)，其中每条文本行都是一个字符串。
为简单起见，我们在这里忽略了标点符号和字母大写。


In [2]:
#@save
# 向 d2l 的全局下载注册表 DATA_HUB 中登记一个数据集，键名为 'time_machine'
# 值的元组形式为 (下载地址, SHA-1 校验和)
#   - 下载地址：由 d2l.DATA_URL 前缀 + 文件名 'timemachine.txt' 拼接而成
#   - SHA-1 ：用于校验文件完整性，确保下载的内容未被损坏或篡改
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')

def read_time_machine():  #@save
    """将时间机器数据集加载到文本行的列表中"""
    # d2l.download 会先检查本地缓存：
    #   - 若已缓存且校验通过，直接返回本地文件路径；
    #   - 否则从网络下载到本地缓存后再返回路径。
    # 随后以只读文本模式 ('r') 打开该文件，with 语句保证用完后自动关闭文件句柄。
    with open(d2l.download('time_machine'), 'r') as f:
        # readlines() 按行读取，返回一个「字符串列表」，每个元素是一行文本
        # （每行末尾会带上换行符 '\n'）
        lines = f.readlines()
    # 使用列表推导式逐行清洗，返回清洗后的字符串列表：
    #   1) re.sub('[^A-Za-z]+', ' ', line)：正则匹配「一个或多个非字母字符」
    #      （数字、标点、空白等都算），把它们统一替换成「单个空格」，
    #      例如 "Hello, world!!" -> "Hello world "
    #   2) .strip()：去掉首尾的空白字符（空格、换行符等）
    #   3) .lower()：全部转为小写，使 "The" 与 "the" 视为同一个词元
    # 结果：每行只保留小写字母和单个空格，简化后续的词元化和词表构建
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

# 调用上面的函数，得到清洗后的文本行列表
lines = read_time_machine()
# 打印文本的总行数，用来确认数据规模
print(f'# 文本总行数: {len(lines)}')
# 打印第 1 行内容，观察清洗后的实际样式
print(lines[0])
# 打印第 11 行内容，进一步确认清洗效果
print(lines[10])

# 文本总行数: 3221
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


### `read_time_machine` 函数详解

一句话总览：

> 这个函数做的事：**下载《时光机器》文本 → 读成按行分开的列表 → 把每一行“洗干净”（只留小写字母和空格）**。

#### 1. 注册数据集（还不会下载）

```python
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')
```

这一步只是**登记信息**，相当于告诉 d2l 三件事：

- 名字叫 `time_machine`
- 从哪里下载（URL = `d2l.DATA_URL` 前缀 + 文件名）
- 下载后用什么校验完整性（SHA-1 校验和）

登记本身不会触发任何网络请求。

#### 2. 下载并打开文件

```python
with open(d2l.download('time_machine'), 'r') as f:
    lines = f.readlines()
```

- `d2l.download('time_machine')`：先查本地缓存，命中就返回本地路径，否则下载后再返回路径。
- `open(路径, 'r')`：以“只读文本”模式打开文件。
- `with ... as f`：`with` 语句保证用完自动关闭文件句柄，无需手动调用 `f.close()`。
- `f.readlines()`：把文件**按行读成一个列表**，每个元素是一整行字符串，且**末尾带着换行符 `\n`**。例如：

```python
lines = [
    "The Time Machine, by H. G. Wells!\n",
    "Chapter I\n",
    "\n",
    ...
]
```

#### 3. 逐行清洗（核心）

```python
return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]
```

这行等价于下面这个循环，先看循环版本更容易理解：

```python
result = []
for line in lines:                              # 遍历每一行
    cleaned = re.sub('[^A-Za-z]+', ' ', line)   # ① 把非字母替换成空格
    cleaned = cleaned.strip()                   # ② 去掉首尾空白
    cleaned = cleaned.lower()                   # ③ 全部转为小写
    result.append(cleaned)                      # 加入新列表
return result
```

`[表达式 for line in lines]` 这种写法叫**列表推导式**，就是把上面的循环压缩成一行。它**不修改**原来的 `lines`，而是返回一个**新列表**。

#### 4. 正则 `re.sub('[^A-Za-z]+', ' ', line)` 拆解

`re.sub` 是 “regular expression substitute”（正则替换），固定吃三个参数：

```python
re.sub(模式, 要替换成什么, 在哪个字符串里找)
#      ↑          ↑            ↑
# '[^A-Za-z]+'  ' '         line
```

它的行为是：在 `line` 里找出**所有**匹配“模式”的片段，把每一个都替换成一个空格，然后返回**新字符串**。

模式 `'[^A-Za-z]+'` 的含义：

| 部分 | 名字 | 含义 |
| --- | --- | --- |
| `[` `]` | 字符集合 | 表示“方括号里这些字符中的某一个” |
| `^` | 在方括号**里面**表示取反 | “不是后面这些东西” |
| `A-Z` | 大写字母 | A 到 Z |
| `a-z` | 小写字母 | a 到 z |
| `+` | 量词 | 前面那个整体重复 **1 次或更多次** |

所以 `[^A-Za-z]+` = **一个或多个连续的、不是字母的字符**。所谓“非字母”包括：空格、逗号、句号、感叹号、数字、换行符 `\n`、下划线等——**除了 26 个大写和 26 个小写字母之外的一切**。

> ⚠️ `^` 有两副面孔：在 `[ ]` **外面**表示“字符串开头”；在 `[ ]` **里面**表示“取反”。这里在括号内，所以是取反。

`+` 的作用很关键：它让连续的非字母被当成**一整段**来匹配，于是 `"!!!"` 只会变成一个空格，而不是三个空格，也就顺便去掉了多余空白。

#### 5. 逐字符追踪一个例子

假设 `line = "The Time Machine, by H.G. Wells!\n"`（`\n` 是行末换行符，也是非字母）：

| 片段 | 类型 | 匹配 `[^A-Za-z]+`？ | 替换为 |
| --- | --- | --- | --- |
| `The` | 字母 | 否 | — |
| `" "` | 空格 | 是 | `" "` |
| `Time` | 字母 | 否 | — |
| `" "` | 空格 | 是 | `" "` |
| `Machine` | 字母 | 否 | — |
| `", "` | 逗号+空格 | 是 | `" "` |
| `by` | 字母 | 否 | — |
| `" "` | 空格 | 是 | `" "` |
| `H` | 字母 | 否 | — |
| `"."` | 句点 | 是 | `" "` |
| `G` | 字母 | 否 | — |
| `"."` | 句点 | 是 | `" "` |
| `" "` | 空格 | 是 | `" "` |
| `Wells` | 字母 | 否 | — |
| `"!\n"` | 感叹号+换行 | 是（一起匹配） | `" "` |

替换后得到 `"The Time Machine by H G Wells "`，再经过 `.strip()` 与 `.lower()`，最终变成：

```
"the time machine by h g wells"
```

#### 6. 两个必须知道的细节

1. **只认英文字母。** `[^A-Za-z]` 的字母集合只包含英文。若 `line` 中含有中文，中文字符也会被当成“非字母”而替换成空格（相当于删掉）。所以这个清洗逻辑是专门给英文语料用的。
2. **会“误伤”一些内容。** 例如 `"don't"` → `"don t"`，`"well-known"` → `"well known"`，`"3.14"` → `" "`。这是有意做的简化取舍：牺牲部分细节，换取干净、紧凑的词元。

#### 7. 最后三行的作用

```python
lines = read_time_machine()
print(f'# 文本总行数: {len(lines)}')
print(lines[0])
print(lines[10])
```

纯粹是打印出来验证效果：确认总行数、观察首行样式、再看第 11 行，检查清洗是否正确。

#### 8. 小结

- `DATA_HUB` 只是**登记**；`download` 才是**下载**。
- `readlines()` 得到**字符串列表**，每行一个元素（带 `\n`）。
- 列表推导式对**每一行**依次执行“替换非字母 → 去首尾空白 → 转小写”。
- `re.sub('[^A-Za-z]+', ' ', line)` 把标点、数字、多余空白都变成**单个空格**；这样后续 `tokenize` 用 `split()` 就能干净地分词，避免 `"machine,"` 与 `"machine"` 被当成两个不同词元。
- `#@save` 是 d2l 书籍的专用标记，表示“把这个函数保存进 d2l 包”，**不影响代码运行**。


## 词元化

下面的`tokenize`函数将文本行列表（`lines`）作为输入，
列表中的每个元素是一个文本序列（如一条文本行）。
[**每个文本序列又被拆分成一个词元列表**]，*词元*（token）是文本的基本单位。
最后，返回一个由词元列表组成的列表，其中的每个词元都是一个字符串（string）。


In [3]:
def tokenize(lines, token='word'):  #@save
    """将文本行拆分为单词或字符词元"""
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('错误：未知词元类型：' + token)

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


### `tokenize` 函数逐行解释

一句话总览：

> 这个函数做的事：**把「每行一个字符串」的列表，变成「每行一个词元列表」的列表**；词元可以是单词（`'word'`）也可以是字符（`'char'`）。

对比一下输入输出的形状：

```python
# 输入：一维的字符串列表（每个元素是一行）
lines = ['the time machine', 'by h g wells']

# 输出：二维的列表（外层每行一个元素，内层是这一行的词元）
# token='word'：
[['the', 'time', 'machine'], ['by', 'h', 'g', 'wells']]
# token='char'：
[['t','h','e',' ','t','i','m','e', ...], ['b','y',' ','h', ...]]
```

---

**第 1 行：**

```python
def tokenize(lines, token='word'):  #@save
```

- 定义一个函数，名字叫 `tokenize`（词元化）。
- 两个参数：
  - `lines`：待处理的文本行列表，每个元素是一个字符串（即上一节 `read_time_machine()` 的返回值）。
  - `token='word'`：**带默认值**的参数。不传时默认按「单词」切分，所以 `tokenize(lines)` 等价于 `tokenize(lines, 'word')`。
- `#@save`：d2l 的专用标记，表示「把这个函数保存进 d2l 包」，**不影响运行**。

**第 2 行：**

```python
    """将文本行拆分为单词或字符词元"""
```

- 这是函数的**文档字符串**（docstring），用来描述函数用途。它不参与运算，只是写给人（和 `help()`）看的。

**第 3～4 行（分支一：按单词切分）：**

```python
    if token == 'word':
        return [line.split() for line in lines]
```

- `if token == 'word':`：判断调用者要的是单词级词元。
- `[line.split() for line in lines]` 是**列表推导式**，等价于：

```python
result = []
for line in lines:          # 逐行遍历
    result.append(line.split())   # 把这一行按空白切成单词列表
return result
```

- `line.split()`：不传参数时，按**任意空白**（空格、制表符、换行符等）切分，并且**自动丢弃连续空白和首尾空白**。
  - `'the time machine'.split()` → `['the', 'time', 'machine']`
  - 正因为上一节已经把标点都换成了空格，这里 `split()` 才能干净地分词。
- 注意**返回的是二维列表**：外层一行一个元素，内层是该行的单词列表。

**第 5～6 行（分支二：按字符切分）：**

```python
    elif token == 'char':
        return [list(line) for line in lines]
```

- `elif` 是「else if」，当前面条件不成立时才检查这里。
- `list(line)`：把一个字符串**打散成单个字符**组成的列表，因为字符串本身可以逐字符迭代。
  - `list('ab c')` → `['a', 'b', ' ', 'c']`
- 所以这一支返回的是：每行 → 该行所有字符的列表。**注意空格也会被当成一个词元保留**。
- 本节最后的 `load_corpus_time_machine` 用的就是字符级（`tokenize(lines, 'char')`）。

**第 7～8 行（兜底分支）：**

```python
    else:
        print('错误：未知词元类型：' + token)
```

- 如果 `token` 既不是 `'word'` 也不是 `'char'`，就打印一条错误提示。
- 这里是「打印」而不是「抛异常」，所以**函数会继续往下走**，最终隐式返回 `None`。这是教程为了简单做的处理；工程代码里更推荐用 `raise ValueError(...)`，让错误更早暴露。
- 字符串拼接 `'错误：未知词元类型：' + token` 能工作，是因为两边都是字符串。

---

**第 10～12 行（调用与验证）：**

```python
tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])
```

- `tokens = tokenize(lines)`：用默认的 `'word'` 模式处理 `lines`，结果是一个二维列表，变量名 `tokens` 沿用全文习惯。
- `range(11)` 生成 `0, 1, ..., 10`，即循环 11 次。
- `print(tokens[i])`：打印第 `i` 行（从 0 开始计数）的分词结果，用来看前 11 行切得对不对。
  - 这里打印的是**列表对象**，所以输出会带方括号和引号，例如 `['the', 'time', 'machine']`。
- 目的纯粹是**验证效果**，不参与后续计算。真正的词表构建在下一节。

---

### 小结

| 行 | 代码 | 作用 |
| --- | --- | --- |
| 1 | `def tokenize(lines, token='word'):` | 定义函数，默认按单词切分 |
| 2 | `"""..."""` | 文档字符串，说明用途 |
| 3-4 | `if token == 'word': return [line.split() ...]` | 按空白切成单词，返回二维列表 |
| 5-6 | `elif token == 'char': return [list(line) ...]` | 按单个字符切分（含空格） |
| 7-8 | `else: print(...)` | 未知类型时打印错误提示 |
| 10 | `tokens = tokenize(lines)` | 实际调用，得到词元列表 |
| 11-12 | `for i in range(11): print(...)` | 打印前 11 行检查结果 |

关键点：

- **输入 1D，输出 2D**：输入是「行列表」，输出是「行的词元列表的列表」。
- `split()` 按空白切分并自动去重空白，所以依赖上一步 `re.sub` 的清洗结果。
- `list(line)` 会把空格也保留为词元，这与 `split()` 的行为不同。
- 默认参数让 `tokenize(lines)` 就是按单词切分。


## 词表

词元的类型是字符串，而模型需要的输入是数字，因此这种类型不方便模型使用。
现在，让我们[**构建一个字典，通常也叫做*词表*（vocabulary），
用来将字符串类型的词元映射到从$0$开始的数字索引中**]。
我们先将训练集中的所有文档合并在一起，对它们的唯一词元进行统计，
得到的统计结果称之为*语料*（corpus）。
然后根据每个唯一词元的出现频率，为其分配一个数字索引。
很少出现的词元通常被移除，这可以降低复杂性。
另外，语料库中不存在或已删除的任何词元都将映射到一个特定的未知词元“&lt;unk&gt;”。
我们可以选择增加一个列表，用于保存那些被保留的词元，
例如：填充词元（“&lt;pad&gt;”）；
序列开始词元（“&lt;bos&gt;”）；
序列结束词元（“&lt;eos&gt;”）。


In [ ]:
class Vocab:  #@save
    """文本词表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        # 按出现频率排序
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                   reverse=True)
                
        # 未知词元的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx
                             for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):  # 未知词元的索引为0
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

def count_corpus(tokens):  #@save
    """统计词元的频率"""
    # 这里的tokens是1D列表或2D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 将词元列表展平成一个列表
        tokens = [token for line in tokens for token in line]
        #actually, the above line is flattening the list of tokens. 
        #If tokens is a list of lists (2D), it will create a single list containing all tokens from all lines. 
        #If tokens is already a 1D list, it will remain unchanged.
    return collections.Counter(tokens)

### `Vocab` 类与 `count_corpus` 函数详解

一句话总览：

> `count_corpus` 负责**数每个词元出现了多少次**；`Vocab` 负责**把词元排好序、编号，并提供「词元 ⇄ 索引」的双向查询**。

先看它俩的分工：

```python
tokens = [['the', 'time', 'machine'], ['by', 'h', 'g', 'wells']]   # 2D 词元列表
count_corpus(tokens)     # Counter({'the':1, 'time':1, 'machine':1, ...})
vocab = Vocab(tokens)    # 把词元映射到 0,1,2,3... 的编号表
vocab['the']             # → 1（查索引）
vocab.to_tokens(1)       # → 'the'（查词元）
```

---

## 一、`count_corpus`：统计词元频率

```python
def count_corpus(tokens):  #@save
    """统计词元的频率"""
    # 这里的tokens是1D列表或2D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 将词元列表展平成一个列表
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)
```

**第 1 行：** `def count_corpus(tokens):` —— 定义函数，参数 `tokens` 既可能是 1D（单行词元列表），也可能是 2D（多行的词元列表的列表）。

**第 3 行：** 判断是否需要「展平」：

| 条件 | 含义 | 为什么 |
| --- | --- | --- |
| `len(tokens) == 0` | 空列表 | 空列表取 `tokens[0]` 会报错，必须先短路掉 |
| `isinstance(tokens[0], list)` | 第一个元素是列表 | 说明这是 2D 结构，需要展平 |

只要满足其一，就把 2D 压成 1D：

```python
tokens = [token for line in tokens for token in line]
```

这是**嵌套列表推导式**，等价于双层循环：

```python
result = []
for line in tokens:        # 外层：遍历每一行
    for token in line:     # 内层：遍历这一行的每个词元
        result.append(token)
```

效果：

```python
[[ 'a','b' ], [ 'c' ]]   →   ['a', 'b', 'c']
```

**第 5 行：** `return collections.Counter(tokens)` —— `Counter` 吃进一个可迭代对象，返回「元素 → 出现次数」的计数结果：

```python
Counter(['a', 'b', 'a'])   # Counter({'a': 2, 'b': 1})
```

> ⚠️ 注意这里的展平是「临时」的：只用局部变量 `tokens` 重新绑定，**不会修改调用者传进来的原列表**。

---

## 二、`Vocab` 类：词元 ⇄ 索引的双向映射

### 0. 三个核心属性

`Vocab` 一创建，内部就维护好三份数据：

| 属性 | 类型 | 方向 | 例子 |
| --- | --- | --- | --- |
| `idx_to_token` | `list` | 索引 → 词元 | `['<unk>', 'the', 'time', ...]` |
| `token_to_idx` | `dict` | 词元 → 索引 | `{'<unk>': 0, 'the': 1, 'time': 2, ...}` |
| `_token_freqs` | `list[tuple]` | 频率排序表 | `[('the', 1000), ('time', 300), ...]` |

`_token_freqs` 前面加下划线，是 Python 的**约定**：暗示「这是内部用的，外面最好通过 `token_freqs` 属性访问」。

### 1. `__init__`：构造函数

```python
def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
```

`__init__` 是 Python 的**构造函数**，`Vocab(tokens)` 创建实例时会自动调用它。三个参数：

- `tokens=None`：语料词元。**默认写成 `None` 而不是 `[]`**，是为了避免「可变默认参数」这个经典坑（默认值只在定义时创建一次，会被多次调用共享）。
- `min_freq=0`：最低频率阈值，出现次数低于它的词元会被丢弃。
- `reserved_tokens=None`：预留的特殊词元，如 `<pad>`、`<bos>`、`<eos>`。

**第 2～5 行**：把 `None` 换成真正的空列表。

```python
if tokens is None:
    tokens = []
if reserved_tokens is None:
    reserved_tokens = []
```

这样即使调用 `Vocab()`（什么都不传），后面也不会因为 `None` 而崩溃。

**第 6～9 行**：统计并**按频率从高到低排序**。

```python
counter = count_corpus(tokens)
self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                           reverse=True)
```

拆开看：

- `counter.items()`：把 `Counter({'a': 2, 'b': 1})` 变成 `[('a', 2), ('b', 1)]`，即「(词元, 频率) 对」的列表。
- `key=lambda x: x[1]`：告诉 `sorted` **按什么排序**。每个元素 `x` 是 `('a', 2)`，`x[1]` 就是频率。`lambda` 是匿名函数，等价于：

```python
def key(x):
    return x[1]
```

- `reverse=True`：降序，高频词排前面。

> 💡 为什么高频词要排前面？因为后面按顺序编号，**高频词拿到小索引**，词表更紧凑、常见词更容易被模型学到。

**第 10～13 行**：先把「未知词元 + 预留词元」放进表里。

```python
self.idx_to_token = ['<unk>'] + reserved_tokens
self.token_to_idx = {token: idx
                     for idx, token in enumerate(self.idx_to_token)}
```

- `['<unk>'] + reserved_tokens`：列表拼接。`<unk>`（unknown，未知）**固定放在索引 0**。
- 字典推导式 `{token: idx for idx, token in enumerate(...)}`：遍历列表，用 `enumerate` 同时拿到「下标」和「词元」，反过来建成「词元 → 下标」的字典。

```python
# 等价于
self.token_to_idx = {}
for idx, token in enumerate(self.idx_to_token):
    self.token_to_idx[token] = idx
```

结果：

```python
self.idx_to_token = ['<unk>', '<pad>', ...]
self.token_to_idx = {'<unk>': 0, '<pad>': 1, ...}
```

**第 14～20 行**：把高频词逐个追加进表。

```python
for token, freq in self._token_freqs:
    if freq < min_freq:
        break
    if token not in self.token_to_idx:
        self.idx_to_token.append(token)
        self.token_to_idx[token] = len(self.idx_to_token) - 1
```

逐行看：

- `for token, freq in self._token_freqs:` —— 元组解包，一次拿到词元和频率。
- `if freq < min_freq: break` —— 因为表已按频率**降序**排列，一旦遇到频率不够的，**后面只会更低**，所以直接 `break` 整段结束，无需 `continue`。这是利用了排序性质的优化。
- `if token not in self.token_to_idx:` —— 防止重复添加（比如词元与 `reserved_tokens` 重名时跳过）。
- `self.idx_to_token.append(token)` —— 加进「索引 → 词元」表。
- `self.token_to_idx[token] = len(self.idx_to_token) - 1` —— 新词元的索引就是**追加后的长度减 1**（即它在列表中的下标）。

> ⚠️ 两张表必须**同步更新**，否则「查索引」和「查词元」会不一致。

### 2. `__len__`：让 `len(vocab)` 可用

```python
def __len__(self):
    return len(self.idx_to_token)
```

这是 Python 的**魔法方法**。实现了它，就能直接写：

```python
len(vocab)     # 等价于 vocab.__len__()
```

返回的是词表大小（词元总个数）。

### 3. `__getitem__`：让 `vocab[...]` 可用

```python
def __getitem__(self, tokens):
    if not isinstance(tokens, (list, tuple)):
        return self.token_to_idx.get(tokens, self.unk)
    return [self.__getitem__(token) for token in tokens]
```

实现了 `__getitem__`，就能用**下标语法** `vocab[x]` 来查索引，这也是 `vocab[tokens[i]]` 能工作的原因。

- 参数名叫 `tokens`（容易让人以为只能传词元），实际它代表「下标里的东西」。
- `isinstance(tokens, (list, tuple))`：判断传进来的是「单个词元（字符串）」还是「一批词元（列表/元组）」。
  - **不是列表/元组**（单个词元）：走 `token_to_idx.get(tokens, self.unk)`。
    - `dict.get(键, 默认值)`：键存在就返回对应值，**不存在就返回默认值**而不报错。
    - 所以「词表里没有的词」统一返回 `self.unk`（即 0），这就是前面说的「未知词元」。
  - **是列表/元组**（一批词元）：递归调用自己处理每个元素，返回索引列表。

```python
vocab['the']                  # → 1
vocab['xyz']                  # → 0（不在词表里，落到 <unk>）
vocab[['the', 'xyz', 'the']]  # → [1, 0, 1]
```

> 💡 用 **递归**而非循环，好处是「单个」和「批量」走同一套逻辑，代码更短；新增嵌套类型（如再套一层 list）也能自动支持。

### 4. `to_tokens`：索引 → 词元

```python
def to_tokens(self, indices):
    if not isinstance(indices, (list, tuple)):
        return self.idx_to_token[indices]
    return [self.idx_to_token[index] for index in indices]
```

结构和 `__getitem__` 完全对称，只是方向相反、查的是另一张表 `idx_to_token`：

```python
vocab.to_tokens(1)          # → 'the'
vocab.to_tokens([1, 0, 2])  # → ['the', '<unk>', 'time']
```

- `idx_to_token` 是列表，所以直接用下标 `[indices]` 取值。
- 如果把「索引 → 词元」也做成字典也可以，但列表按索引取值是 $O(1)$，且更省内存，所以这里用列表最合适。

> 🔎 为什么 `__getitem__` 用 `get(...)` 容错，而 `to_tokens` 不用？因为「未知词元 → 0」是合理且必要的；但「未知索引 → 某个词元」没有对错之分，直接让 `IndexError` 暴露出来更好。

### 5. `@property` 属性：`unk` 与 `token_freqs`

```python
@property
def unk(self):  # 未知词元的索引为0
    return 0

@property
def token_freqs(self):
    return self._token_freqs
```

`@property` 是**装饰器**，把方法「伪装」成属性。于是可以像属性一样访问，而不用加括号：

```python
vocab.unk          # ✅ 0
vocab.unk()        # ❌ 会报错，0 不是可调用的
vocab.token_freqs  # ✅ 返回频率排序表
```

好处：

- **只读**：外部无法 `vocab.unk = 5` 去改坏它，起到保护作用。
- **语法统一**：使用方不用关心它是「存好的字段」还是「现算出来的」。
- `unk` 返回常量 0，是因为 `<unk>` 恒定占据索引 0，写成属性比到处硬编码 `0` 更清晰（`self.token_to_idx.get(tokens, self.unk)`）。

---

## 三、几个容易踩的细节

1. **`<unk>` 的索引一定是 0**。因为 `idx_to_token` 初始化为 `['<unk>'] + reserved_tokens`，所以 `unk` 属性直接 `return 0`。
2. **`break` 而非 `continue`**。依赖 `_token_freqs` 的降序排列，遇到低于阈值直接终止循环。
3. **`Reserved tokens` 会占用 1 之后的索引**。由于 `min_freq` 过滤是在预留词元之后进行的，预留词元**不受 `min_freq` 影响**。
4. **传入 2D 词元列表没问题**。`count_corpus` 会自动展平，所以 `Vocab(tokens)` 里 `tokens` 是几维都行（但不能是空列表以外的奇怪结构）。
5. **`min_freq` 会显著影响词表大小**。这也是本节练习 2 想让你实验的点：`min_freq` 调大 → 低频词被丢弃 → 词表更小、`<unk>` 更多。

---

## 四、小结

| 成员 | 作用 |
| --- | --- |
| `count_corpus(tokens)` | 展平 2D 词元列表并用 `Counter` 统计频率 |
| `Vocab.__init__` | 统计 → 降序排序 → 建 `idx_to_token` / `token_to_idx` 双向表 |
| `Vocab.__len__` | 支持 `len(vocab)`，返回词表大小 |
| `Vocab.__getitem__` | 支持 `vocab[...]`，词元 → 索引，未知词元落到 0 |
| `Vocab.to_tokens` | 索引 → 词元，方向相反 |
| `Vocab.unk` | 未知词元索引（恒为 0） |
| `Vocab.token_freqs` | 按频率降序的 `(词元, 频率)` 列表 |

一句话记住：**`idx_to_token` 是列表（索引找词），`token_to_idx` 是字典（词找索引），`_token_freqs` 是排序依据；三者都由 `__init__` 一次性建好，之后只读。**

### 附：`sorted` 与 `list.sort()` 用法速查

本节用到的排序语句是：

```python
self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                           reverse=True)
```

下面把 Python 的两套排序 API 完整梳理一遍。

---

## 一、两个 API 的区别

| | `sorted(可迭代对象)` | `列表.sort()` |
| --- | --- | --- |
| 类型 | **内置函数** | **列表的方法** |
| 返回值 | **新列表** | `None` |
| 原数据 | 不变 | **就地修改** |
| 适用范围 | 任何可迭代对象（`dict`、`tuple`、生成器都可以） | 只能是列表 |

```python
nums = [3, 1, 2]

new = sorted(nums)     # new = [1, 2, 3]，nums 仍是 [3, 1, 2]
ret = nums.sort()      # ret = None，nums 变成 [1, 2, 3]（已被改动）
```

> ⚠️ 最常见的坑：`nums = nums.sort()` 会让 `nums` 变成 `None`。因为 `sort()` **返回 `None`**。
>
> 同理，**元组、字符串不能 `.sort()`**（不可变），只能用 `sorted()`。

## 二、两个关键字参数

```python
sorted(iterable, *, key=None, reverse=False)
```

| 参数 | 作用 | 默认 |
| --- | --- | --- |
| `key` | 一个函数，**对每个元素先算出「排序依据」**，再按依据比较 | `None`（直接比较元素本身） |
| `reverse` | `True` 表示**降序** | `False`（升序） |

注意 `key` 和 `reverse` 前有 `*`，它们是**只允许按名字传**的参数：

```python
sorted(x, key=len)        # ✅
sorted(x, len)            # ❌ TypeError
```

## 三、`key` 的四种常见写法

假设有这批数据：

```python
words = ['Banana', 'apple', 'Cherry']
pairs = [('the', 100), ('time', 30), ('a', 100)]
people = [('Tom', 20), ('Amy', 30)]
```

**① 用内置函数**（把函数名本身当参数传，不加括号）：

```python
sorted(words, key=len)          # 按长度：['apple', 'Banana', 'Cherry']
sorted(words, key=str.lower)    # 忽略大小写的字典序：['apple', 'Banana', 'Cherry']
sorted([-3, 1, -2], key=abs)    # 按绝对值：[1, -2, -3]
```

> 💡 写 `key=len` 而不是 `key=len()`。传的是「函数本身」，`sorted` 会在内部逐个调用它。

**② 用 `lambda`**（临场定义，适合「取第几个元素」「做点小计算」）：

```python
sorted(pairs, key=lambda x: x[1])                    # 按频率，升序
sorted(pairs, key=lambda x: x[1], reverse=True)      # 按频率，降序（本节的写法）
```

`lambda x: x[1]` 等价于：

```python
def key(x):
    return x[1]
```

**③ 用 `operator.itemgetter` / `attrgetter`**（取下标 / 取属性的快捷写法）：

```python
from operator import itemgetter, attrgetter

sorted(pairs, key=itemgetter(1))          # 等价于 key=lambda x: x[1]
sorted(pairs, key=itemgetter(1, 0))       # 先按 x[1]，再按 x[0]
sorted(objs, key=attrgetter('freq'))      # 按对象的 .freq 属性
```

**④ 用 `key` 返回元组做「多级排序」**：

```python
# 先按频率降序（这里先整体升序），频率相同时按词元的字典序
sorted(pairs, key=lambda x: (x[1], x[0]))
```

因为元组是**逐位比较**的：先比第 1 位，相同再比第 2 位，以此类推。

## 四、升降序混用的技巧

`sorted` 只接受一个 `reverse`，无法对「第 1 位降序、第 2 位升序」这种混合需求直接表达。两种常见绕法：

**方法 A：取负号**（仅适用于数值）

```python
# 频率降序，词元升序
sorted(pairs, key=lambda x: (-x[1], x[0]))
```

**方法 B：分两次排**（利用稳定性，**先排次要的，再排主要的**）

```python
# 先按词元升序
r = sorted(pairs, key=lambda x: x[0])
# 再按频率降序（稳定的，会保留上一步的词元顺序）
r = sorted(r, key=lambda x: x[1], reverse=True)
```

> 🔑 这是「稳定排序」最实用的用法：**多趟排序，从最次要的键排到最主要的键**。

## 五、稳定排序（stability）

排序是**稳定**的，意思是：**比较结果相等的元素，会保持它们在原始序列中的相对次序**。

```python
data = [('a', 1), ('b', 1), ('c', 0)]
sorted(data, key=lambda x: x[1])
# [('c', 0), ('a', 1), ('b', 1)]
#              ↑ 'a' 仍在 'b' 前面，因为原来就是这样
```

本节里的表现：`Counter` 内部按「词元第一次出现」记录顺序，所以**频率相同的词元，先出现的排在前面**。

## 六、常见「坑」清单

| 代码 | 问题 | 正确写法 |
| --- | --- | --- |
| `x = x.sort()` | `x` 变成 `None` | `x.sort()` 或 `x = sorted(x)` |
| `key=len()` | 把 `len()` 的结果当依据 | `key=len` |
| `sorted(['10', '9'])` | 字符串按字典序，得 `['10', '9']` | 转成数字：`key=int` |
| `sorted([[1,2],[3]])` | 长度不同的列表比较会报错/结果反直觉 | 用 `key=len` 等明确依据 |
| `sorted({'a':2, 'b':1})` | 对 `dict` 排序得到的是**键**的列表 | 要键值对用 `d.items()` |
| `sorted(d.items(), key=lambda x: x[1])` | 忘了 `reverse=True` | 降序要显式写 `reverse=True` |

## 七、回到本节那一行

```python
counter.items()                                # dict_items([('the', 100), ('time', 30), ...])
sorted(..., key=lambda x: x[1], reverse=True)  # 按 x[1]（频率）降序
```

| 部分 | 作用 |
| --- | --- |
| `counter.items()` | 得到 `(词元, 频率)` 元组序列 |
| `key=lambda x: x[1]` | 比较依据取「频率」，即 `x[1]` |
| `reverse=True` | 高频在前 |
| 赋值给 `self._token_freqs` | `sorted` 返回**新列表**，存起来供后续 `for` 循环按序编号 |

等价写法（`Counter` 自带方法）：

```python
self._token_freqs = counter.most_common()    # 就是「按频率降序」的意思
```

## 八、速查表

```python
sorted(x)                                   # 升序，元素本身比较
sorted(x, reverse=True)                     # 降序
sorted(x, key=len)                          # 按长度
sorted(x, key=str.lower)                    # 忽略大小写
sorted(x, key=lambda e: e[1])               # 按第 2 个分量
sorted(x, key=lambda e: (e[1], e[0]))       # 多级：先 e[1] 再 e[0]
sorted(x, key=lambda e: (-e[1], e[0]))      # 混合升降序（数值）
sorted(x, key=itemgetter(1))                # 同 lambda e: e[1]，略快
sorted(x, key=attrgetter('name'))           # 按对象属性
x.sort(key=..., reverse=...)                # 就地排序，返回 None
```

我们首先使用时光机器数据集作为语料库来[**构建词表**]，然后打印前几个高频词元及其索引。


In [5]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

[('<unk>', 0), ('the', 1), ('i', 2), ('and', 3), ('of', 4), ('a', 5), ('to', 6), ('was', 7), ('in', 8), ('that', 9)]


现在，我们可以(**将每一条文本行转换成一个数字索引列表**)。


In [6]:
for i in [0, 10]:
    print('文本:', tokens[i])
    print('索引:', vocab[tokens[i]])

文本: ['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
索引: [1, 19, 50, 40, 2183, 2184, 400]
文本: ['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']
索引: [2186, 3, 25, 1044, 362, 113, 7, 1421, 3, 1045, 1]


## 整合所有功能

在使用上述函数时，我们[**将所有功能打包到`load_corpus_time_machine`函数中**]，
该函数返回`corpus`（词元索引列表）和`vocab`（时光机器语料库的词表）。
我们在这里所做的改变是：

1. 为了简化后面章节中的训练，我们使用字符（而不是单词）实现文本词元化；
1. 时光机器数据集中的每个文本行不一定是一个句子或一个段落，还可能是一个单词，因此返回的`corpus`仅处理为单个列表，而不是使用多词元列表构成的一个列表。


In [9]:
def load_corpus_time_machine(max_tokens=-1):  #@save
    """返回时光机器数据集的词元索引列表和词表"""
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    # 因为时光机器数据集中的每个文本行不一定是一个句子或一个段落，
    # 所以将所有文本行展平到一个列表中
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)

(170580, 28)

## 小结

* 文本是序列数据的一种最常见的形式之一。
* 为了对文本进行预处理，我们通常将文本拆分为词元，构建词表将词元字符串映射为数字索引，并将文本数据转换为词元索引以供模型操作。

## 练习

1. 词元化是一个关键的预处理步骤，它因语言而异。尝试找到另外三种常用的词元化文本的方法。
1. 在本节的实验中，将文本词元为单词和更改`Vocab`实例的`min_freq`参数。这对词表大小有何影响？


[Discussions](https://discuss.d2l.ai/t/2094)
